In [1]:
import pandas as pd
import numpy as np
import json

def generar_anclas_estadisticas(csv_path):
    print("📊 Generando diccionario de anclas estadísticas...")
    df = pd.read_csv(csv_path)
    
    # Filtramos valores atípicos extremos para no ensuciar la estadística
    # (Por ejemplo, productos con confidence muy bajo)
    if 'confidence' in df.columns:
        df = df[df['confidence'] >= 0.85]

    diccionario_anclas = {}
    
    # Agrupamos por los factores más determinantes de la base del precio
    grupos = df.groupby(['category', 'subtype', 'market_tier', 'is_premium_brand'])
    
    for nombre_grupo, datos_grupo in grupos:
        if len(datos_grupo) < 3:
            continue # Ignoramos micro-segmentos con muy pocos datos
            
        cat, subtype, tier, premium = nombre_grupo
        
        # Calculamos en el espacio logarítmico (log_original_price)
        mediana_log = datos_grupo['log_original_price'].median()
        
        # Calculamos cuánto suele variar el precio en este segmento (MAD)
        # MAD (Median Absolute Deviation) es más resistente a outliers que la desviación estándar
        mad_log = np.median(np.abs(datos_grupo['log_original_price'] - mediana_log))
        
        clave = f"{cat}|{subtype}|{tier}|{premium}"
        diccionario_anclas[clave] = {
            "mediana_log": float(mediana_log),
            "mad_log": float(mad_log) if mad_log > 0.05 else 0.05 # Límite mínimo de tolerancia
        }

    # Guardamos el cerebro estadístico en un JSON
    with open('estadisticas_precio.json', 'w') as f:
        json.dump(diccionario_anclas, f, indent=4)
        
    print(f"✅ ¡Guardadas {len(diccionario_anclas)} anclas de micro-segmentos en 'estadisticas_precio.json'!")

# Ejecutar con el dataset de producción (sin los 12 del test)
generar_anclas_estadisticas("../../../Datasets/evaluacion4_produccion.csv")


📊 Generando diccionario de anclas estadísticas...
✅ ¡Guardadas 188 anclas de micro-segmentos en 'estadisticas_precio.json'!
